In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Step 1: Load and preprocess dataset
df = pd.read_csv("Churn_Prediction_Preprocessed.csv")
df = pd.get_dummies(df, drop_first=True)
X = df.drop("churn_flag", axis=1)
y = df["churn_flag"]

In [3]:
print("Original number of features:", X.shape[1])

Original number of features: 1164


In [4]:
# Step 2: Remove low variance features
vt = VarianceThreshold(threshold=0.01)
X_reduced = vt.fit_transform(X)
X_columns_reduced = X.columns[vt.get_support()]
X = pd.DataFrame(X_reduced, columns=X_columns_reduced)
print(f"After variance threshold: {X.shape}")

After variance threshold: (5000, 41)


In [5]:
# Step 3: Sample dataset if too large
if X.shape[0] > 10000:
    sample_idx = np.random.choice(X.index, size=5000, replace=False)
    X_sample = X.loc[sample_idx]
    y_sample = y.loc[sample_idx]
else:
    X_sample = X
    y_sample = y

In [6]:
# Step 4: Model switch for feature selection
use_model = "random_forest"  # options: "logistic", "decision_tree", "random_forest"

if use_model == "logistic":
    print("Using Logistic Regression for feature selection...")
    estimator = LogisticRegression(max_iter=1000)
elif use_model == "decision_tree":
    print("Using Decision Tree for feature selection...")
    estimator = DecisionTreeClassifier(random_state=0)
elif use_model == "random_forest":
    print("Using Random Forest for feature selection...")
    estimator = RandomForestClassifier(n_estimators=100, random_state=0)
else:
    raise ValueError("Invalid model selection for feature selection")

# Perform backward feature selection
selector = SequentialFeatureSelector(
    estimator,
    n_features_to_select=6,
    direction="backward",
    cv=3,
    n_jobs=-1
)
selector.fit(X_sample, y_sample)

selected_features = X.columns[selector.get_support()]
X_selected = X[selected_features]
print("Selected features:", list(selected_features))

Using Random Forest for feature selection...
Selected features: ['auto_renew_flag', 'Licensed user count', 'industry_EdTech', 'referral_source_event', 'referral_source_other', 'referral_source_partner']


In [ ]:
# Step 5: Train-test split and scale
def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_and_scale(X_selected, y)

In [ ]:
# Step 6: Evaluation function with GridSearchCV
def evaluate_model_with_tuning(model, param_grid, name):
    print(f"\n🔍 Tuning {name}...")
    grid_search = GridSearchCV(model, param_grid, cv=3, n_jobs=-1, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Best Accuracy: {acc:.4f}")
    print(f"Best Parameters: {grid_search.best_params_}")
    
    return {
        "Model": name,
        "Accuracy": acc,
        "Best Params": grid_search.best_params_,
        "Report": classification_report(y_test, y_pred, output_dict=False),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

In [ ]:
# Step 7: Define hyperparameter grids
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"]
    },
    "SVM (Linear)": {
        "C": [0.1, 1, 10]
    },
    "SVM (RBF)": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.001]
    },
    "Naive Bayes": {
        # No params for GaussianNB
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"]
    },
    "Decision Tree": {
        "max_depth": [3, 5, 10, None],
        "criterion": ["gini", "entropy"]
    },
    "Random Forest": {
        "n_estimators": [50, 100],
        "max_depth": [5, 10, None],
        "criterion": ["gini", "entropy"]
    }
}


In [ ]:
# Step 8: Run all models
results = []

results.append(evaluate_model_with_tuning(LogisticRegression(max_iter=1000), param_grids["Logistic Regression"], "Logistic Regression"))
results.append(evaluate_model_with_tuning(SVC(kernel='linear'), param_grids["SVM (Linear)"], "SVM (Linear)"))
results.append(evaluate_model_with_tuning(SVC(kernel='rbf'), param_grids["SVM (RBF)"], "SVM (RBF)"))
results.append(evaluate_model_with_tuning(GaussianNB(), {}, "Naive Bayes"))  # No params to tune
results.append(evaluate_model_with_tuning(KNeighborsClassifier(), param_grids["KNN"], "KNN"))
results.append(evaluate_model_with_tuning(DecisionTreeClassifier(), param_grids["Decision Tree"], "Decision Tree"))
results.append(evaluate_model_with_tuning(RandomForestClassifier(), param_grids["Random Forest"], "Random Forest"))

In [ ]:
# Step 9: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"}).sort_values(by="Accuracy", ascending=False)

print("\n📊 Model Accuracy Comparison After Hyperparameter Tuning for Random Forest:")
print(summary_df)

In [ ]:
# Step 9: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"}).sort_values(by="Accuracy", ascending=False)

print("\n📊 Model Accuracy Comparison After Hyperparameter Tuning for Logistic regression:")
print(summary_df)

In [ ]:
# Step 9: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"}).sort_values(by="Accuracy", ascending=False)

print("\n📊 Model Accuracy Comparison After Hyperparameter Tuning for Decision Tree:")
print(summary_df)